In [9]:
# Python Library
import os
import glob
import sys
import numpy as np

from astropy.coordinates import SkyCoord
from astropy.time import Time
from astropy import units as u
from astropy.io import fits
from astropy.table import Table
from astropy.table import vstack
from astropy.table import hstack
import warnings
warnings.filterwarnings("ignore")

# Plot presetting
import matplotlib.pyplot as plt
import matplotlib as mpl

# Jupyter Setting
mpl.rcParams["axes.titlesize"] = 14
mpl.rcParams["axes.labelsize"] = 20
plt.rcParams['savefig.dpi'] = 500
plt.rc('font', family='serif')

In [10]:
### Helper Functions
import sys
sys.path.append(os.path.join('..', 'src'))
from helper import makeSpecColors
from paths import *
from var import *
from sdtpy import *

In [11]:
logtxt = ""

In [12]:
input_rubin_table = os.path.join(AUGMNETED_PHOT_RUBIN_DETECTED_DATA, "synphot_normal_class_detection.csv")	
input_7dt_table = os.path.join(AUGMNETED_PHOT_7DT_DETECTED_DATA, "synphot_normal_class_detection.csv")	

rubin_table = Table.read(input_rubin_table)
sdt_table = Table.read(input_7dt_table)

print(f"Rubin Table: {len(rubin_table):,} rows")
print(f"7DT Table: {len(sdt_table):,} rows")

logtxt += f"Rubin Table: {len(rubin_table):,} rows\n"
logtxt += f"7DT Table: {len(sdt_table):,} rows\n"

Rubin Table: 87,270 rows
7DT Table: 84,560 rows


In [13]:
import pandas as pd
from astropy.table import Table

# 1) Astropy → pandas
df_rubin = rubin_table.to_pandas().reset_index().rename(columns={'index':'rubin_idx'})
df_sdt   = sdt_table.to_pandas().reset_index().rename(columns={'index':'sdt_idx'})

# 2) 키 컬럼만 추출해서 inner join → 공통 키만 남긴다
common = pd.merge(
    df_sdt[['spec','n_iter','scale_factor']], 
    df_rubin[['spec','n_iter','scale_factor']],
    on=['spec','n_iter','scale_factor'],
    how='inner'
)

# 3) 이 공통 키를 기준으로 원본 테이블에서 필터링
df_rubin_common = pd.merge(
    df_rubin,
    common,
    on=['spec','n_iter','scale_factor'],
    how='inner'
)
df_sdt_common = pd.merge(
    df_sdt,
    common,
    on=['spec','n_iter','scale_factor'],
    how='inner'
)

# 4) 필요하면 다시 Astropy Table로!
rubin_common_table = Table.from_pandas(df_rubin_common.drop(columns=['rubin_idx']))
sdt_common_table   = Table.from_pandas(df_sdt_common  .drop(columns=['sdt_idx']))

logtxt += f"{len(rubin_common_table):,} rows\n"
logtxt += f"{len(sdt_common_table):,} rows\n"

In [14]:

logtxt += f"Number Check: {len(rubin_common_table):,} == {len(sdt_common_table):,} ({len(rubin_common_table) == len(sdt_common_table)})\n"
len(rubin_common_table) == len(sdt_common_table)

True

In [15]:
keys_common = list(set(rubin_common_table.keys()) & set(sdt_common_table.keys()))
key_ignore = [f"{key}_{i}" for key in keys_common for i in (1, 2)] + [
    key for key in sdt_common_table.keys() if any(x in key for x in ("n_iter", "scale_factor", "fnu", "magapp", "magerr", "snr", "magabs"))
] + [
    key for key in rubin_common_table.keys() if any(x in key for x in ("n_iter", "scale_factor", "fnu", "magapp", "magerr", "snr", "magabs"))
]

print(keys_common)
print(key_ignore)

['scale_factor', 'type', 'flagtype', 'detection', 'n_iter', 'spec']
['scale_factor_1', 'scale_factor_2', 'type_1', 'type_2', 'flagtype_1', 'flagtype_2', 'detection_1', 'detection_2', 'n_iter_1', 'n_iter_2', 'spec_1', 'spec_2', 'magabs_m400', 'snr_m400', 'magapp_m400', 'magerr_m400', 'fnuobs_m400', 'fnuerr_m400', 'fnu_m400', 'n_iter_m400', 'scale_factor_m400', 'magabs_m412', 'snr_m412', 'magapp_m412', 'magerr_m412', 'fnuobs_m412', 'fnuerr_m412', 'fnu_m412', 'n_iter_m412', 'scale_factor_m412', 'magabs_m425', 'snr_m425', 'magapp_m425', 'magerr_m425', 'fnuobs_m425', 'fnuerr_m425', 'fnu_m425', 'n_iter_m425', 'scale_factor_m425', 'magabs_m437', 'snr_m437', 'magapp_m437', 'magerr_m437', 'fnuobs_m437', 'fnuerr_m437', 'fnu_m437', 'n_iter_m437', 'scale_factor_m437', 'magabs_m450', 'snr_m450', 'magapp_m450', 'magerr_m450', 'fnuobs_m450', 'fnuerr_m450', 'fnu_m450', 'n_iter_m450', 'scale_factor_m450', 'magabs_m462', 'snr_m462', 'magapp_m462', 'magerr_m462', 'fnuobs_m462', 'fnuerr_m462', 'fnu_m462',

In [16]:
rubin_keys_to_merge = keys_common + [key for key in rubin_common_table.keys() if key not in keys_common and key not in key_ignore]
sdt_keys_to_merge = [key for key in sdt_common_table.keys() if key not in keys_common and key not in key_ignore]

merged_table = hstack([rubin_common_table[rubin_keys_to_merge], sdt_common_table[sdt_keys_to_merge]])
logtxt += f"Number of merged synphot tables: {len(merged_table):,}\n"
merged_table[:3]

scale_factor,type,flagtype,detection,n_iter,spec,magobs_u,magobs_g,magobs_r,magobs_i,magobs_z,magobs_y,magobs_m400,magobs_m412,magobs_m425,magobs_m437,magobs_m450,magobs_m462,magobs_m475,magobs_m487,magobs_m500,magobs_m512,magobs_m525,magobs_m537,magobs_m550,magobs_m562,magobs_m575,magobs_m587,magobs_m600,magobs_m612,magobs_m625,magobs_m637,magobs_m650,magobs_m662,magobs_m675,magobs_m687,magobs_m700,magobs_m712,magobs_m725,magobs_m737,magobs_m750,magobs_m762,magobs_m775,magobs_m787,magobs_m800,magobs_m812,magobs_m825,magobs_m837,magobs_m850,magobs_m862,magobs_m875,magobs_m887
float64,str4,str12,str4,int64,str160,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64
0.1,Ia,aug_blue_red,True,0,/Users/paek/Research/7DT/SED-Classifier/notebook/../data/Spectra/OSC/tPS1-15p_20150114_Gr13_Free_slit1.0_1_f.asci,20.483498053489885,19.945039087218575,19.85252914191766,20.060525654368888,20.174016442505515,20.171441790585487,20.141783494477206,19.825760046150563,19.785162730979557,20.00616357700131,20.01935189461504,19.845429309230127,19.81045509615514,19.98001748024605,20.159122087758547,20.12063773298775,19.809451629343382,19.912401790757723,20.055952697766017,19.896536369798916,19.853712871281587,19.89732332336138,19.772102938933536,19.989141596659515,20.285582725450432,20.05265853725569,19.76928929553758,19.48050811969315,19.775780132833027,19.917306682392976,19.908219098571564,20.061031528353777,20.03910090063478,19.89932446585576,19.956556407568765,20.235705003033225,20.340623200439207,19.607743176854328,19.84661946260311,20.302155445272042,20.473795019313975,20.84610119480447,20.605266612340472,19.18139634772309,19.617312910286625,19.833967624568945
1.0,Ia,aug_blue_red,True,0,/Users/paek/Research/7DT/SED-Classifier/notebook/../data/Spectra/OSC/tPS1-15p_20150114_Gr13_Free_slit1.0_1_f.asci,17.98465245618083,17.447431117618162,17.35081595055412,17.548592941003136,17.666732447390583,17.64530520011839,17.608097311649985,17.319971402251362,17.342905129773857,17.56943325719613,17.577582183039656,17.334017880278044,17.328805138971994,17.468823880181066,17.656049960043685,17.5778187010107,17.32997919027993,17.414277385313216,17.550860325253957,17.412906833241518,17.260804693902237,17.307918839758035,17.291362354141302,17.396647051120272,17.758707217573306,17.569705291435422,17.153420302279113,17.120281172898522,17.25225683766319,17.400864712050396,17.482851124515637,17.494818275961396,17.506257999744694,17.474849803109723,17.541081283147367,17.722481214542842,17.67328468448334,17.56855278174115,17.450076742490886,17.658447103851806,17.773772858298194,17.729245649783138,17.746510187444517,17.673669289730654,17.402598066434756,17.44193890245583
10.0,Ia,aug_blue_red,True,0,/Users/paek/Research/7DT/SED-Classifier/notebook/../data/Spectra/OSC/tPS1-15p_20150114_Gr13_Free_slit1.0_1_f.asci,15.48315808108587,14.947496967394343,14.850604500987489,15.048272460815905,15.167238224136518,15.145416063801232,15.109720412948231,14.827314407721678,14.843836127619827,15.066535172782823,15.058380451192779,14.83052848912466,14.814567062139497,14.98283567585275,15.14129639870962,15.06012482901226,14.84545303389059,14.910713001670166,15.045822074523453,14.9188902398302,14.777773279942908,14.791120400965612,14.807264469517799,14.88429051670477,15.283094495722661,15.04596023432356,14.63101591425726,14.632355227392127,14.762733709605168,14.884908428901017,14.952904515648427,14.984044104972597,15.004115387945621,14.974112000890647,15.032997362722401,15.225454217413201,15.198530573214668,14.998864049565722,15.004572650231163,15.178180124927001,15.283505051374581,15.301387569248911,15.328765430684863,15.109769516268848,14.919350729203872,14.989193936928865


In [17]:
merged_table_name = os.path.join(AUGMNETED_PHOT_DATA, "merged_synphot_normal_class.csv")
merged_table.write(merged_table_name, format='csv', overwrite=True)
print(f"Merged table saved to {merged_table_name}")
logtxt += f"Merged table saved to {merged_table_name}\n"
logtxt += "\n"
logtxt += "END\n"
print(logtxt)

Merged table saved to /Users/paek/Research/7DT/SED-Classifier/notebook/../data/Synphot/Augmented_Photometry/merged_synphot_normal_class.csv
Rubin Table: 87,270 rows
7DT Table: 84,560 rows
81,925 rows
81,925 rows
Number Check: 81,925 == 81,925 (True)
Number of merged synphot tables: 81,925
Merged table saved to /Users/paek/Research/7DT/SED-Classifier/notebook/../data/Synphot/Augmented_Photometry/merged_synphot_normal_class.csv

END



In [18]:
logtxt_path = os.path.join(AUGMNETED_PHOT_DATA, "log.txt")
with open(logtxt_path, 'w') as f:
    f.write(logtxt)